In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import mean, min, max, stddev, col, date_format, to_json, create_map, lit, year, month, dayofmonth, collect_set

@dp.materialized_view(
    name="dev_emp.gold.employee_behavior",
    comment="Gold layer: Employee behavior metrics aggregated to one row per account_id with statistical measures"
)
def employee_behavior():
    """
    Read the silver table and aggregate employee behavior metrics.
    Creates one row per account_id with day, week, hour, and daytime statistics.
    """
    # Read from silver table
    df5 = spark.read.table("dev_emp.silver.employee_day_behaviour")
    
    active = df5.select('date', 'dayname', 'week', 'hour', 'daytime', 'account_id', 'messages_per_day', 'messages_per_hour',
                     'max_hourly_messages_per_day', 'messages_per_week', 'messages_per_daytime')
    # print("DONE")
    # Week Calculation( Need to get clarified on the week calculation)
    df2_week = active.groupBy('date', 'week', 'account_id') \
        .agg(
        mean(col('messages_per_week')).alias('MEAN_count_of_messages_per_week'),
        min(col('messages_per_week')).alias('MIN_count_of_messages_per_week'),
        max(col('messages_per_week')).alias('MAX_count_of_messages_per_week'),
        stddev(col('messages_per_week')).alias('STDDEV_count_of_messages_per_week')
    ) \
        .withColumn("dayname", date_format(col('date'), "EEEE"))

    ##calculation on dayname
    ######
    df2_dayname = active.groupBy('date', 'dayname', 'account_id').agg(
        mean(col('messages_per_week')).alias('MEAN_count_of_messages_per_week'),
        min(col('messages_per_week')).alias('MIN_count_of_messages_per_week'),
        max(col('messages_per_week')).alias('MAX_count_of_messages_per_week'),
        stddev(col('messages_per_week')).alias('STDDEV_count_of_messages_per_week'),
        mean(col('messages_per_day')).alias('MEAN_count_of_messages_per_day'),
        min(col('messages_per_day')).alias('MIN_count_of_messages_per_day'),
        max(col('messages_per_day')).alias('MAX_count_of_messages_per_day'),
        stddev(col('messages_per_day')).alias('STDDEV_count_of_messages_per_day'),
        mean(col('messages_per_hour')).alias('MEAN_count_of_messages_per_hour'),
        min(col('messages_per_hour')).alias('MIN_count_of_messages_per_hour'),
        max(col('messages_per_hour')).alias('MAX_count_of_messages_per_hour'),
        stddev(col('messages_per_hour')).alias('STDDEV_count_of_messages_per_hour'),
        mean(col('max_hourly_messages_per_day')).alias('MEAN_max_hourly_messages_per_day'),
        min(col('max_hourly_messages_per_day')).alias('MIN_max_hourly_messages_per_day'),
        max(col('max_hourly_messages_per_day')).alias('MAX_max_hourly_messages_per_day'),
        stddev(col('max_hourly_messages_per_day')).alias('STDDEV_max_hourly_messages_per_day'))

    ### Calculation on daytime
    df2_daytime = active.groupBy('date', 'dayname', 'daytime', 'account_id').agg(
        mean(col('messages_per_daytime')).alias('MEAN_count_of_messages_per_daytime'),
        min(col('messages_per_daytime')).alias('MIN_count_of_messages_per_daytime'),
        max(col('messages_per_daytime')).alias('MAX_count_of_messages_per_daytime'),
        stddev(col('messages_per_daytime')).alias('STDDEV_count_of_messages_per_daytime'))

    # Calculation on bm_messages_per_day,bm_messages_per_hour,bm_max_hourly_messages_per_day
    bm_day = df2_dayname.withColumn('bm_messages_per_day', to_json(
        create_map(lit('mean'), col('MEAN_count_of_messages_per_day'), lit('max'), col('MAX_count_of_messages_per_day'),
                   lit('min'), col('MIN_count_of_messages_per_day'), lit('std'),
                   col('STDDEV_count_of_messages_per_day')))) \
        .withColumn('bm_messages_per_hour', to_json(
        create_map(lit('mean'), col('MEAN_count_of_messages_per_hour'), lit('max'), col('MAX_count_of_messages_per_hour'),
                   lit('min'), col('MIN_count_of_messages_per_hour'), lit('std'),
                   col('STDDEV_count_of_messages_per_hour')))) \
        .withColumn('bm_max_hourly_messages_per_day', to_json(create_map(lit('mean'),
                                                                         col('MEAN_max_hourly_messages_per_day'),
                                                                         lit('max'),
                                                                         col('MAX_max_hourly_messages_per_day'), lit('min'),
                                                                         col('MIN_max_hourly_messages_per_day'), lit('std'),
                                                                         col('STDDEV_max_hourly_messages_per_day'))))

    ## Calculation on bm_messages_per_daytime
    bm_daytime = df2_daytime.withColumn('bm_messages_per_daytime', to_json(
        create_map(lit('mean'), col('MEAN_count_of_messages_per_daytime'), lit('max'),
                   col('MAX_count_of_messages_per_daytime'), lit('min'), col('MIN_count_of_messages_per_daytime'),
                   lit('std'), col('STDDEV_count_of_messages_per_daytime'))))

    #####Calculation on bm_messages_per_week####
    ###
    bm_week = df2_week.withColumn('bm_messages_per_week', to_json(
        create_map(lit('mean'), col('MEAN_count_of_messages_per_week'), lit('max'), col('MAX_count_of_messages_per_week'),
                   lit('min'), col('MIN_count_of_messages_per_week'), lit('std'),
                   col('STDDEV_count_of_messages_per_week'))))

    bm = bm_day.join(bm_daytime, on=['date', 'account_id', 'dayname']) \
        .join(bm_week, on=['date', 'account_id', 'dayname']) \
        .withColumn('map_dn_max_hourly_messages_per_specific_day', to_json(create_map(col('dayname'),
                                                                                      create_map(lit('mean'),
                                                                                                 col('MEAN_max_hourly_messages_per_day'),
                                                                                                 lit('max'),
                                                                                                 col('MAX_max_hourly_messages_per_day'),
                                                                                                 lit('min'),
                                                                                                 col('MIN_max_hourly_messages_per_day'),
                                                                                                 lit('std'),
                                                                                                 col('STDDEV_max_hourly_messages_per_day'))))) \
        .withColumn('map_dt_messages_per_specific_daytime', to_json(create_map(col('daytime'), create_map(lit('mean'),
                                                                                                          col('MEAN_count_of_messages_per_daytime'),
                                                                                                          lit('max'),
                                                                                                          col('MAX_count_of_messages_per_daytime'),
                                                                                                          lit('min'),
                                                                                                          col('MIN_count_of_messages_per_daytime'),
                                                                                                          lit('std'),
                                                                                                          col('STDDEV_count_of_messages_per_daytime'))))) \
        .withColumn('map_dn_messages_per_specific_day', to_json(create_map(col('dayname'), create_map(lit('mean'),
                                                                                                      col('MEAN_count_of_messages_per_day'),
                                                                                                      lit('max'),
                                                                                                      col('MAX_count_of_messages_per_day'),
                                                                                                      lit('min'),
                                                                                                      col('MIN_count_of_messages_per_day'),
                                                                                                      lit('std'),
                                                                                                      col('STDDEV_count_of_messages_per_day'))))) \
        .withColumn("year", year(col('date').cast("timestamp"))).withColumn('month', month(
        col('date').cast('timestamp'))).withColumn("day", dayofmonth(col('date').cast('timestamp')))
    df4 = bm.groupBy('account_id', 'date', 'dayname').agg(
        collect_set('map_dn_messages_per_specific_day').alias('dn_messages_per_specific_day'),
        collect_set('map_dn_max_hourly_messages_per_specific_day').alias('dn_max_hourly_messages_per_specific_day'),
        collect_set('map_dt_messages_per_specific_daytime').alias('dt_messages_per_specific_daytime'))
    df5 = bm.join(df4, on=['account_id', 'date', 'dayname'], how='inner').drop('Interval', 'count',
                                                                               'map_dn_messages_per_specific_day',
                                                                               'map_dn_max_hourly_messages_per_specific_day',
                                                                               'map_dt_messages_per_specific_daytime')
    df6 = df5.select('account_id', 'daytime', 'dayname', 'bm_messages_per_day', 'bm_messages_per_hour',
                     'bm_messages_per_daytime', 'bm_messages_per_week', 'bm_max_hourly_messages_per_day',
                     'dn_messages_per_specific_day', 'dn_max_hourly_messages_per_specific_day',
                     'dt_messages_per_specific_daytime', 'year', 'month', 'day')
    
    return df6